# Phase 6 v3 — version 33: full-coverage long-text summarization, LoRA → distillation
**Kaggle:** select **GPU T4 ×2**, enable Internet, attach the `phase2v3` dataset, then **Save & Run All**. Kaggle assigns its own saved-version number; v33 labels this deliverable.

**Pipeline:** exact Phase 2 v3 MLA/ALiBi base + v2 tokenizer → overlapping source windows → chunk-aligned LoRA teacher training → online knowledge distillation into a two-thirds-depth student → both models summarize **every** source chunk → merge/deduplicate generated sentences into one document summary → one prediction-only submission per model.

Each prompt contains at most 640 tokens including instructions; adjacent source chunks overlap by **128 tokens**. Targets use up to 384 tokens, so training stays inside the original 1024-token context. No source window is silently skipped and there is no input-length truncation. Very long documents require proportionally more computation; full coverage does not guarantee that every fact survives the final 768-token summary.

**No external RAG:** the complete judgment is already available. No vector database, embedding model, external corpus or test-label retrieval is added. Chunking is a direct document summarization solution. During training only, reference sentences are weakly aligned to their best source chunk with source-local IDF cosine similarity. Reference text is never used to choose test chunks or inference inputs. This is heuristic alignment, not human chunk labels.

**Quality changes:** train on chunk-relevant targets instead of repeating the entire reference for every chunk; no fake EOS at truncated targets; rank-32 LoRA over MLA/MLP projections and norms; two-thirds-depth student; time-aware LR decay; document-level deduplication and compression of generated sentences. The reducer does not call an untrained second-pass model or substitute copied source text. These are targeted changes, not proof of better real-data quality. The run prints base/teacher/student ROUGE using the **same 16 held-out documents and identical chunking/reduction**, plus actual optimizer updates and validation CE.

**Saved predictions only:**
```text
/kaggle/working/phase6_v3_v33_predictions/
├── submission_lora/
│   └── prediction.json
├── submission_lora.zip       # contains only prediction.json
├── submission_student/
│   └── prediction.json
└── submission_student.zip    # contains only prediction.json
```
Weights, caches, diagnostics and shards are temporary in `/tmp/phase6_v3_v33_runtime`. They are removed after both ZIPs verify. No weights are exported; the input checkpoint is unchanged. Metrics remain in notebook output.

**Runtime:** generation is benchmarked on both GPUs; the reserve accounts for the **actual number of chunks and per-chunk decoding budgets** for both models. Remaining time is allocated to LoRA and KD. All chunks are processed; if the full-coverage workload cannot fit, the run fails clearly instead of dropping chunks. The target remains an 86-minute internal budget and 88-minute subprocess cap, leaving overhead before 90 minutes. Actual Kaggle throughput and the genuine-data full run have not been verified in this authoring environment.


In [ ]:
# Version 33 settings and end-to-end clock
import time, json, os, tempfile
from pathlib import Path
RUN_STARTED_AT = time.time()
CONFIG = dict(
    version="phase6-v3-v33-longtext", seed=67,
    checkpoint="/kaggle/input/datasets/koooshiieee/phase2v3/final_model (1).pt",
    tokenizer_repo="DumbLad/legal-bpe-tokenizer-phase1-v2",
    tokenizer_dir=str(Path(tempfile.gettempdir())/"phase6_v3_v33_runtime"/"tokenizer"),
    dataset_repo="Exploration-Lab/CS-781-Capstone",
    train_remote="Phase_6_Summ/Train/train.json", test_remote="Phase_6_Summ/Test/test.json",
    out_dir=str(Path(tempfile.gettempdir())/"phase6_v3_v33_runtime"),
    submission_dir="/kaggle/working/phase6_v3_v33_predictions", cleanup_runtime=False,
    input_tokens=640, target_tokens=384, chunk_overlap=128,
    summary_tokens=768, min_chunk_new_tokens=48, max_chunk_new_tokens=192,
    alignment_min_similarity=0.08,
    rank=32, alpha=64, dropout=0.05,
    lora_lr=3e-4, student_lr=2e-5, student_depth_fraction=2/3,
    micro_batch=2, grad_accum=2, gen_batch=8,
    teacher_steps=1600, student_steps=1200, min_updates=20,
    eval_every=75, val_loss_docs=32, val_loss_chunks=32, val_gen_docs=16,
    temperature=2.0, ce_weight=0.7, kd_positions=64,
    total_minutes=86, started_at=RUN_STARTED_AT,
)
# Predictions only in final outputs; checkpoints/caches remain temporary and are deleted on success.
# Optional HF secret: HF, or environment HF_TOKEN, for downloads only.
assert Path(CONFIG['checkpoint']).is_file(), f"Attach phase2v3; missing {CONFIG['checkpoint']}"
os.makedirs(CONFIG['out_dir'], exist_ok=True)
Path('phase6_v3_v33_config.json').write_text(json.dumps(CONFIG, indent=2))


In [ ]:
# Install lightweight dependencies and verify two GPUs
%pip -q install huggingface_hub regex rouge-score tqdm
import torch, subprocess, sys, signal
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Choose accelerator GPU T4 x2 before Save & Run All."
print('PyTorch', torch.__version__)
for gpu in range(2): print(gpu, torch.cuda.get_device_name(gpu))
if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF')
    except Exception:
        print('No HF secret: trying public downloads.')

def run_checked(args, log_name):
    remaining=RUN_STARTED_AT+88*60-time.time()
    assert remaining>0, '88-minute execution deadline reached.'
    log_path=Path(CONFIG['out_dir'])/log_name
    # Inherit stdout so Kaggle shows live training progress; tee to a file using a reader thread.
    import threading
    with log_path.open('w',encoding='utf8') as log:
        p=subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
        def stream():
            for line in p.stdout:
                log.write(line); log.flush(); print(line,end='',flush=True)
        thread=threading.Thread(target=stream,daemon=True);thread.start()
        try: rc=p.wait(timeout=remaining)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid,signal.SIGTERM);p.wait();thread.join(timeout=5)
            raise RuntimeError(f'Runtime deadline reached in {log_name}; inspect partial artifacts. No completed submission is claimed.')
        thread.join(timeout=10)
    assert rc==0, f'Process failed ({rc}); see {log_path}'


Exact copy from Phase 2 v3: cell beginning `%%writefile llama_scratch.py`. Do not retain the old GQA-only LM definition from Phase 6 v1.

In [ ]:
%%writefile llama_scratch.py
"""
Llama-style decoder-only LM, written from scratch (no HF / no llama library).

Pieces: RMSNorm, RoPE (or ALiBi), grouped-query attention (GQA; MQA = n_kv_heads 1,
MHA = n_kv_heads == n_heads), SwiGLU MLP, pre-norm residual blocks, tied embeddings,
KV-cache generation with left-padded batches.

Only torch tensors/autograd/nn.Linear/nn.Embedding are used as building blocks.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------- config
@dataclass
class Config:
    vocab_size: int = 33000
    d_model: int = 1024
    n_layers: int = 16
    n_heads: int = 16
    n_kv_heads: int = 4          # 16 query heads share 4 KV heads (4 queries per KV head)
    ffn_hidden: int = 2560
    max_seq_len: int = 1024      # training context
    rope_cache_len: int = 4096   # RoPE table length (>= prompt + generated tokens)
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    tie_embeddings: bool = True
    pos_enc: str = "rope"        # "rope" | "alibi"
    attn_type: str = "gqa"       # "gqa" | "mla"
    kv_latent: int = 128         # MLA only: size of the cached K/V latent
    attn_impl: str = "manual"    # "manual" (hand-written softmax(QK^T)V) | "sdpa" (torch fused kernel, same math)


PRESETS = {
    # exact totals with vocab 33000, tied embeddings
    "v2_100m": dict(d_model=768, n_layers=12, n_heads=12, n_kv_heads=4, ffn_hidden=2048), # ~100.86M
    "target_150m": dict(d_model=896, n_layers=14, n_heads=14, n_kv_heads=2, ffn_hidden=2304), # ~142M 
    "base":  dict(d_model=1024, n_layers=16, n_heads=16, n_kv_heads=4, ffn_hidden=2560),  # ~201.6M
    "small": dict(d_model=1024, n_layers=12, n_heads=16, n_kv_heads=4, ffn_hidden=2816),  # ~169.1M
    "v3_100m_mla": dict(d_model=768, n_layers=12, n_heads=12, n_kv_heads=12, ffn_hidden=1920, attn_type="mla", kv_latent=256),  # ~99.7M
    "v3_80m_mla": dict(d_model=640, n_layers=12, n_heads=10, n_kv_heads=10, ffn_hidden=1984, attn_type="mla", kv_latent=160),  # ~80.4M
    "v3_80m_gqa": dict(d_model=640, n_layers=12, n_heads=10, n_kv_heads=2, ffn_hidden=2048),  # ~80.1M, control
    "tiny":  dict(d_model=64,   n_layers=2,  n_heads=4,  n_kv_heads=2, ffn_hidden=128),   # smoke tests
}


def analytic_param_count(c: Config) -> int:
    """Closed-form parameter count (use it to size a model before building it)."""
    hd = c.d_model // c.n_heads
    if c.attn_type == "mla":
        attn = 2* c.d_model* c.n_heads*hd + c.d_model*c.kv_latent+c.kv_latent\
               + 2 * c.kv_latent * c.n_heads * hd        # Wq,Wo + W_dkv + kv_norm + W_uk,W_uv
    else:
        attn = 2 * c.d_model * c.n_heads * hd + 2 * c.d_model * c.n_kv_heads * hd
    mlp = 3 * c.d_model * c.ffn_hidden                                           # gate, up, down
    per_layer = attn + mlp + 2 * c.d_model                                       # + 2 RMSNorm
    total = per_layer * c.n_layers + c.vocab_size * c.d_model + c.d_model        # + embedding + final norm
    if not c.tie_embeddings:
        total += c.vocab_size * c.d_model
    return total


def count_params(model, verbose=True) -> int:
    """model.parameters() yields a tied (shared) tensor only once, so tied weights are counted once."""
    total = sum(p.numel() for p in model.parameters())
    if verbose:
        emb = model.tok_emb.weight.numel()
        blocks = sum(p.numel() for p in model.layers.parameters())
        print(f"total {total/1e6:.1f}M | transformer blocks {blocks/1e6:.1f}M | embedding {emb/1e6:.1f}M "
              f"({'tied with output head' if model.cfg.tie_embeddings else 'output head separate'}) "
              f"| analytic {analytic_param_count(model.cfg)/1e6:.1f}M")
    return total


# ----------------------------------------------------------------------------- RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()                                   # always normalise in fp32
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)


# ----------------------------------------------------------------------------- RoPE
def build_rope_cache(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))   # (hd/2,)
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                                                # (max_len, hd/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: (B, H, T, hd); cos/sin: (T, hd/2). Rotates the two halves of each head vector (Llama layout)."""
    half = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :half], xf[..., half:]
    cos = cos[None, None]
    sin = sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


# ----------------------------------------------------------------------------- ALiBi
def get_alibi_slopes(n_heads):
    def pow2(n):
        start = 2 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_heads).is_integer():
        s = pow2(n_heads)
    else:
        c = 2 ** math.floor(math.log2(n_heads))
        s = pow2(c) + pow2(2 * c)[0::2][: n_heads - c]
    return torch.tensor(s, dtype=torch.float32)


# ----------------------------------------------------------------------------- GQA
def repeat_kv(x, n_rep):
    """(B, n_kv, S, hd) -> (B, n_kv*n_rep, S, hd); query head h uses kv head h // n_rep."""
    if n_rep == 1:
        return x
    B, n_kv, S, hd = x.shape
    return x[:, :, None].expand(B, n_kv, n_rep, S, hd).reshape(B, n_kv * n_rep, S, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.nh, self.nkv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.n_rep = self.nh // self.nkv
        self.impl = cfg.attn_impl
        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)    # (B, nh,  T, hd)
        k = self.wk(x).view(B, T, self.nkv, self.hd).transpose(1, 2)   # (B, nkv, T, hd)
        v = self.wv(x).view(B, T, self.nkv, self.hd).transpose(1, 2)

        if cos is not None:                                            # RoPE on q and k (not v)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past is not None:                                           # cache holds only nkv heads
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        new_cache = (k, v) if use_cache else None
        S = k.shape[2]

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # relative distance of every (query, key) pair; keys j <= query position are visible
        qpos = torch.arange(S - T, S, device=x.device)
        kpos = torch.arange(S, device=x.device)
        rel = qpos[:, None] - kpos[None, :]                            # (T, S)
        allowed = (rel >= 0)[None, None]                               # (1, 1, T, S)
        if key_valid is not None:                                      # hide left-padding keys
            allowed = allowed & key_valid[:, None, None, :]
        bias = None
        if alibi_slopes is not None:
            bias = -alibi_slopes.view(1, -1, 1, 1) * rel.clamp(min=0).float()[None, None]   # (1, nh, T, S)

        if self.impl == "sdpa" and key_valid is None and T == S:      # training fast path
            if bias is None:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            else:
                m = bias.masked_fill(~allowed, torch.finfo(q.dtype).min).to(q.dtype)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        else:                                                          # hand-written attention
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()   # (B, nh, T, S)
            if bias is not None:
                scores = scores + bias
            scores = scores.masked_fill(~allowed, -1e30)               # finite, so a fully-masked pad row can't give NaN
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v                                            # (B, nh, T, hd)

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache

# ----------------------------------------------------------------------------- MLA
class MLAttention(nn.Module):
    """Simplified multi-head latent attention. K and V are rebuilt from one small latent per token;
    only the latent (B, S, kv_latent) is cached. ALiBi only."""
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        assert cfg.pos_enc == "alibi", "this MLA has no decoupled RoPE; use --pos_enc alibi"
        self.nh = cfg.n_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.impl = cfg.attn_impl
        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.w_dkv = nn.Linear(cfg.d_model, cfg.kv_latent, bias=False)
        self.kv_norm = RMSNorm(cfg.kv_latent, cfg.norm_eps)
        self.w_uk = nn.Linear(cfg.kv_latent, self.nh * self.hd, bias=False)
        self.w_uv = nn.Linear(cfg.kv_latent, self.nh * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        c = self.kv_norm(self.w_dkv(x))                               # (B, T, r)
        if past is not None:
            c = torch.cat([past, c], dim=1)                           # cache is just the latent
        new_cache = c if use_cache else None
        S = c.shape[1]
        k = self.w_uk(c).view(B, S, self.nh, self.hd).transpose(1, 2)
        v = self.w_uv(c).view(B, S, self.nh, self.hd).transpose(1, 2)

        qpos = torch.arange(S - T, S, device=x.device)
        kpos = torch.arange(S, device=x.device)
        rel = qpos[:, None] - kpos[None, :]
        allowed = (rel >= 0)[None, None]
        if key_valid is not None:
            allowed = allowed & key_valid[:, None, None, :]
        bias = None
        if alibi_slopes is not None:
            bias = -alibi_slopes.view(1, -1, 1, 1) * rel.clamp(min=0).float()[None, None]

        if self.impl == "sdpa" and key_valid is None and T == S:
            if bias is None:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            else:
                m = bias.masked_fill(~allowed, torch.finfo(q.dtype).min).to(q.dtype)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        else:
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()
            if bias is not None:
                scores = scores + bias
            scores = scores.masked_fill(~allowed, -1e30)
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache

# ----------------------------------------------------------------------------- SwiGLU
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


# ----------------------------------------------------------------------------- block + model
class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = MLAttention(cfg) if cfg.attn_type == "mla" else GQAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.ffn_hidden)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        h, new_cache = self.attn(self.attn_norm(x), cos, sin, alibi_slopes, key_valid, past, use_cache)
        x = x + h
        x = x + self.mlp(self.mlp_norm(x))
        return x, new_cache


class LlamaLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.pos_enc in ("rope", "alibi") and cfg.attn_impl in ("manual", "sdpa")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        if cfg.pos_enc == "rope":
            cos, sin = build_rope_cache(cfg.d_model // cfg.n_heads, cfg.rope_cache_len, cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
            self.alibi_slopes = None
        else:
            self.rope_cos = self.rope_sin = None
            self.register_buffer("alibi_slopes", get_alibi_slopes(cfg.n_heads), persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():       # GPT-2 style: shrink the residual-branch output projections
            if name.endswith("wo.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False, cache=None, start_pos=0,
                key_valid=None, all_logits=False):
        """
        training:  model(x, targets=y) -> scalar loss
        inference: model(idx, use_cache=True, cache=..., start_pos=..., key_valid=...) -> (logits, cache)
                   logits are for the last position only unless all_logits=True.
        """
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos = sin = None
        if self.cfg.pos_enc == "rope":
            assert start_pos + T <= self.cfg.rope_cache_len, "sequence longer than the RoPE table"
            cos = self.rope_cos[start_pos:start_pos + T]
            sin = self.rope_sin[start_pos:start_pos + T]

        new_caches = []
        for i, layer in enumerate(self.layers):
            past = cache[i] if cache is not None else None
            x, c = layer(x, cos, sin, self.alibi_slopes, key_valid, past, use_cache)
            new_caches.append(c)
        x = self.norm(x)

        if targets is not None:
            logits = self.lm_head(x)
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        logits = self.lm_head(x if all_logits else x[:, -1:, :])
        return logits, (new_caches if use_cache else None)


# ----------------------------------------------------------------------------- generation
@torch.no_grad()
def generate(model, prompts, max_new_tokens, eos_id, pad_id, temperature=0.1, top_k=30,
             top_p=0.95, rep_penalty=1.2, ban_eos=True):
    """
    prompts: list of lists of token ids (any lengths). Left-pads them into one batch, runs one prefill,
    then decodes with a KV cache. Returns a list of lists of generated ids (EOS / padding removed).
    temperature <= 0 -> greedy.
    """
    model.eval()
    device = next(model.parameters()).device
    V = model.cfg.vocab_size
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    key_valid = torch.zeros(B, L, dtype=torch.bool, device=device)
    seen = torch.zeros(B, V, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        t = torch.tensor(p, dtype=torch.long, device=device)
        ids[b, L - len(p):] = t
        key_valid[b, L - len(p):] = True
        seen[b, t] = True

    out = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    amp = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda"))
    with amp:
        logits, cache = model(ids, use_cache=True, key_valid=key_valid, start_pos=0)
        for step in range(max_new_tokens):
            lg = logits[:, -1, :].float()
            if ban_eos:
                lg[:, eos_id] = float("-inf")
            if rep_penalty != 1.0:                                     # CTRL-style repetition penalty
                pen = torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty)
                lg = torch.where(seen, pen, lg)
            if temperature <= 0:
                nxt = lg.argmax(-1)
            else:
                lg = lg / temperature
                if top_k and top_k > 0:
                    kth = torch.topk(lg, min(top_k, V)).values[:, -1:]
                    lg = lg.masked_fill(lg < kth, float("-inf"))
                if top_p < 1.0:
                    sl, si = torch.sort(lg, descending=True, dim=-1)
                    pr = torch.softmax(sl, dim=-1)
                    drop = (pr.cumsum(-1) - pr) > top_p                # keep the token that crosses top_p
                    sl = sl.masked_fill(drop, float("-inf"))
                    lg = torch.full_like(lg, float("-inf")).scatter(1, si, sl)
                nxt = torch.multinomial(torch.softmax(lg, dim=-1), 1).squeeze(1)

            nxt = torch.where(finished, torch.full_like(nxt, pad_id), nxt)
            for b in range(B):
                if not finished[b] and nxt[b].item() != eos_id:
                    out[b].append(int(nxt[b].item()))
            finished = finished | (nxt == eos_id)
            seen.scatter_(1, nxt[:, None], True)
            if bool(finished.all()):
                break
            key_valid = torch.cat([key_valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
            logits, cache = model(nxt[:, None], use_cache=True, cache=cache,
                                  start_pos=L + step, key_valid=key_valid)
    return out

Exact copy from Phase 2 v3: cell beginning `%%writefile bpe_tokenizer.py`. The download repository must be its **v2 tokenizer**.

In [ ]:
%%writefile bpe_tokenizer.py
#!/usr/bin/env python3
import os, json, regex
from typing import Dict, List, Tuple
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import regex


_PAT = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


class PureByteLevelBPETokenizer:
    def __init__(self, vocab_size: int = 33000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]

        self.encoder: Dict[bytes, int] = {}
        self.decoder: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], bytes] = {}       # pair -> merged token
        self.merge_ranks: Dict[Tuple[bytes, bytes], int] = {}    # pair -> order learned (lower = earlier)
        self._cache: Dict[str, List[int]] = {}

        self._init_base_vocab()

    def _init_base_vocab(self):
        for b in range(256):
            b_bytes = bytes([b])
            self.encoder[b_bytes] = b
            self.decoder[b] = b_bytes

        # special tokens
        for idx, token in enumerate(self.special_tokens, start=256):
            b_bytes = token.encode("utf-8")
            self.encoder[b_bytes] = idx
            self.decoder[idx] = b_bytes

    @property
    def pad_token_id(self) -> int:
        return self.encoder["<pad>".encode("utf-8")]

    @property
    def unk_token_id(self) -> int:
        return self.encoder["<unk>".encode("utf-8")]

    @property
    def bos_token_id(self) -> int:
        return self.encoder["<s>".encode("utf-8")]

    @property
    def eos_token_id(self) -> int:
        return self.encoder["</s>".encode("utf-8")]

    # ------------------------------------------------------------------ training
    @staticmethod
    def _merge_word(word, a, b, new):
        out, i, n = [], 0, len(word)
        while i < n:
            if i < n - 1 and word[i] == a and word[i + 1] == b:
                out.append(new)
                i += 2
            else:
                out.append(word[i])
                i += 1
        return out

    def train_from_texts(self, texts: List[str], target_vocab_size: int = None, min_word_freq: int = 2):
        if target_vocab_size is None:
            target_vocab_size = self.vocab_size
        num_merges = target_vocab_size - len(self.encoder)
        print(f"\nTraining byte level BPE from scratch. Target merges: {num_merges:,} | Base Vocab: {len(self.encoder)}")

        # 1) pre-tokenize + count unique words (the BPE then runs on this small table, not the raw text)
        word_counts = Counter()
        for text in tqdm(texts, desc="Counting pre-tokens"):
            if text and text.strip():
                word_counts.update(_PAT.findall(text))

        words, freqs = [], []
        for w, c in word_counts.items():
            if c >= min_word_freq:
                words.append([bytes([b]) for b in w.encode("utf-8", "replace")])
                freqs.append(c)
        del word_counts
        print(f"Unique words used for merge learning: {len(words):,}")

        # 2) initial pair statistics (updated incrementally afterwards)
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for i, w in enumerate(tqdm(words, desc="Initial pair counts")):
            for p in zip(w, w[1:]):
                pair_counts[p] += freqs[i]
                where[p].add(i)

        # 3) merge loop
        pbar = tqdm(total=num_merges, desc="Learning BPE Merges", dynamic_ncols=True)
        for _ in range(num_merges):
            if not pair_counts:
                print("No more candidate pairs found to merge.")
                break

            best_pair, freq = max(pair_counts.items(), key=lambda kv: kv[1])
            if freq < 2:
                print(f"Stopping early: Most frequent pair frequency is {freq}")
                break

            a, b = best_pair
            merged_token = a + b
            self.merges[best_pair] = merged_token
            self.merge_ranks[best_pair] = len(self.merge_ranks)
            if merged_token not in self.encoder:          # two merge paths can give the same bytes
                new_id = len(self.encoder)
                self.encoder[merged_token] = new_id
                self.decoder[new_id] = merged_token

            for i in list(where[best_pair]):
                w, f = words[i], freqs[i]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        del pair_counts[p]
                w = self._merge_word(w, a, b, merged_token)
                words[i] = w
                for p in zip(w, w[1:]):
                    pair_counts[p] += f
                    where[p].add(i)
            del where[best_pair]
            pair_counts.pop(best_pair, None)

            pbar.update(1)
            pbar.set_postfix({"vocab_size": len(self.encoder), "top_pair_freq": freq})

        pbar.close()
        self._cache.clear()
        print(f"Training finished! Final Vocab Size: {len(self.encoder):,}")

    # ------------------------------------------------------------------ encode / decode
    def _encode_word(self, word: str) -> List[int]:
        cached = self._cache.get(word)
        if cached is not None:
            return cached
        tokens = [bytes([b]) for b in word.encode("utf-8", "replace")]
        while len(tokens) >= 2:
            best_pair, best_rank = None, None
            for p in zip(tokens, tokens[1:]):
                r = self.merge_ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_pair, best_rank = p, r
            if best_pair is None:
                break
            tokens = self._merge_word(tokens, best_pair[0], best_pair[1], self.merges[best_pair])
        ids = [self.encoder.get(t, self.unk_token_id) for t in tokens]
        if len(self._cache) < 1_000_000:
            self._cache[word] = ids
        return ids

    def encode(self, text: str) -> List[int]:
        ids: List[int] = []
        for word in _PAT.findall(text):
            ids.extend(self._encode_word(word))
        return ids

    def decode(self, token_ids: List[int]) -> str:
        raw_bytes = b"".join(self.decoder.get(i, b"<unk>") for i in token_ids)
        return raw_bytes.decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ save / load
    def save(self, save_dir: str):
        os.makedirs(save_dir, exist_ok=True)
        vocab_path = os.path.join(save_dir, "vocab.json")
        merges_path = os.path.join(save_dir, "merges.json")
        config_path = os.path.join(save_dir, "tokenizer_config.json")

        # latin1 maps bytes 1:1 onto code points, so any byte string survives JSON
        ser_vocab = {k.decode("latin1"): v for k, v in self.encoder.items()}
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(ser_vocab, f, ensure_ascii=False, indent=2)

        # JSON (not a space-separated text file): tokens can contain spaces, newlines, \r, \xa0 ...
        ser_merges = [[p1.decode("latin1"), p2.decode("latin1")] for (p1, p2) in self.merges]  # in learned order
        with open(merges_path, "w", encoding="utf-8") as f:
            json.dump(ser_merges, f, ensure_ascii=False)

        config = {
            "tokenizer_class": "PureByteLevelBPETokenizer",
            "vocab_size": len(self.encoder),
            "special_tokens": self.special_tokens,
            "pad_token_id": self.pad_token_id,
            "bos_token_id": self.bos_token_id,
            "eos_token_id": self.eos_token_id
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

        print(f"Tokenizer saved successfully to {save_dir}")

    @classmethod
    def load(cls, save_dir: str) -> "PureByteLevelBPETokenizer":
        tok = cls()
        with open(os.path.join(save_dir, "vocab.json"), "r", encoding="utf-8") as f:
            ser_vocab = json.load(f)
        tok.encoder = {k.encode("latin1"): v for k, v in ser_vocab.items()}
        tok.decoder = {v: k.encode("latin1") for k, v in ser_vocab.items()}

        tok.merges, tok.merge_ranks = {}, {}
        with open(os.path.join(save_dir, "merges.json"), "r", encoding="utf-8") as f:
            for rank, (s1, s2) in enumerate(json.load(f)):
                p1, p2 = s1.encode("latin1"), s2.encode("latin1")
                tok.merges[(p1, p2)] = p1 + p2
                tok.merge_ranks[(p1, p2)] = rank
        return tok


Shared model helpers, overlapping full-coverage windows, reference-to-chunk alignment, LoRA/KD objectives, chunk batching, generated-sentence reduction and completeness checks.

In [ ]:
%%writefile phase6_v3_common.py
import os, json, math, random, re, time, hashlib
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import asdict
from llama_scratch import Config, LlamaLM
from bpe_tokenizer import PureByteLevelBPETokenizer

def read_config():
    return json.loads(Path('phase6_v3_v33_config.json').read_text())

def seed_all(seed):
    random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def atomic_json(obj, path):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf8'); tmp.replace(path)

def save_model(model, path, **meta):
    path=Path(path); tmp=path.with_suffix('.tmp')
    torch.save(dict(cfg=asdict(model.cfg), model={k:v.detach().cpu() for k,v in model.state_dict().items()}, **meta),tmp)
    tmp.replace(path)

class LoRALinear(nn.Module):
    def __init__(self, base, rank=16, alpha=32, dropout=.05):
        super().__init__(); self.base=base; self.scale=alpha/rank
        self.A=nn.Parameter(torch.empty(rank,base.in_features)); self.B=nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5)); self.drop=nn.Dropout(dropout)
    def forward(self,x):
        return self.base(x)+F.linear(F.linear(self.drop(x),self.A),self.B)*self.scale

def inject_lora(model,c):
    model.requires_grad_(False); names=[]
    for i,b in enumerate(model.layers):
        targets=['wq','w_dkv','w_uk','w_uv','wo'] if model.cfg.attn_type=='mla' else ['wq','wk','wv','wo']
        for parent,attrs in [(b.attn,targets),(b.mlp,['w_gate','w_up','w_down'])]:
            for name in attrs:
                setattr(parent,name,LoRALinear(getattr(parent,name),c['rank'],c['alpha'],c['dropout']))
                names.append(f'layers.{i}.{"attn" if parent is b.attn else "mlp"}.{name}')
    # Norms adapt cheaply; embeddings stay frozen (no giant embedding optimizer states).
    for name,p in model.named_parameters():
        if 'norm' in name: p.requires_grad_(True)
    return names

def merge_lora(model):
    for parent in list(model.modules()):
        for name,m in list(parent.named_children()):
            if isinstance(m,LoRALinear):
                with torch.no_grad(): m.base.weight.add_((m.B@m.A)*m.scale)
                setattr(parent,name,m.base)
    return model

def load_base(c):
    ck=torch.load(c['checkpoint'],map_location='cpu',weights_only=False)
    assert isinstance(ck,dict) and 'cfg' in ck and 'model' in ck, 'Expected original LM checkpoint containing cfg and model, not a Phase-2 NER head.'
    cfg=Config(**ck['cfg']); cfg.attn_impl='sdpa'
    assert cfg.attn_type=='mla' and cfg.pos_enc=='alibi', f'Wrong v3 base architecture: {cfg}'
    assert c['input_tokens']+c['target_tokens']<=cfg.max_seq_len, 'Token budget exceeds original training context.'
    m=LlamaLM(cfg); m.load_state_dict(ck['model'],strict=True)
    assert all(torch.isfinite(p).all() for p in m.parameters()), 'Nonfinite base weights'
    return m

def load_saved(path):
    ck=torch.load(path,map_location='cpu',weights_only=False)
    m=LlamaLM(Config(**ck['cfg'])); m.load_state_dict(ck['model'],strict=True); return m

def student_from_teacher(teacher,c):
    tc=asdict(teacher.cfg); n=tc['n_layers']; k=max(2,min(n-1,round(n*c.get('student_depth_fraction',2/3))))
    selected=[round(i*(n-1)/(k-1)) for i in range(k)]
    tc['n_layers']=k; student=LlamaLM(Config(**tc)); sd=student.state_dict(); ts=teacher.state_dict()
    for name in sd:
        if name.startswith('layers.'):
            a=name.split('.'); a[1]=str(selected[int(a[1])]); source='.'.join(a)
        else: source=name
        sd[name]=ts[source].clone()
    student.load_state_dict(sd,strict=True); student.requires_grad_(True)
    return student,selected

def hidden(model,x):
    h=model.tok_emb(x); co=si=None
    if model.cfg.pos_enc=='rope': co,si=model.rope_cos[:x.shape[1]],model.rope_sin[:x.shape[1]]
    # Right padding needs no key mask: causal attention means real tokens cannot see later pads.
    for b in model.layers: h,_=b(h,co,si,model.alibi_slopes,None,None,False)
    return model.norm(h)

class Objective(nn.Module):
    def __init__(self,model): super().__init__(); self.model=model
    def forward(self,x,y,kd_positions=None):
        h=hidden(self.model,x); mask=y.ne(-100); z=self.model.lm_head(h[mask]).float()
        ce=F.cross_entropy(z,y[mask])
        return ce, (z[kd_positions] if kd_positions is not None else z[:0])

def collate(examples,pad,device):
    length=max(len(e['x']) for e in examples)
    x=torch.full((len(examples),length),pad,dtype=torch.long); y=torch.full_like(x,-100)
    for i,e in enumerate(examples):
        x[i,:len(e['x'])]=torch.tensor(e['x']); y[i,:len(e['y'])]=torch.tensor(e['y'])
    return x.to(device),y.to(device)

def token_ids(tok,text):
    ids=tok.encode(text); return ids.ids if hasattr(ids,'ids') else ids

def make_target(tok,text,limit):
    raw=token_ids(tok,text)
    # A cropped reference is not an actual end of summary: do not teach false early EOS.
    return (raw+[tok.eos_token_id] if len(raw)<limit else raw[:limit]), len(raw)>=limit

def chunk_document(tok,text,c):
    prefix=token_ids(tok,'Legal judgment excerpt:\n')
    suffix=token_ids(tok,'\nSummarize the important facts, legal reasoning and outcome stated in this excerpt only:\n')
    budget=c['input_tokens']-1-len(prefix)-len(suffix)
    overlap=c['chunk_overlap']; assert 0<=overlap<budget and budget>64
    ids=token_ids(tok,text); result=[]; start=0
    while start<len(ids):
        end=min(start+budget,len(ids))
        result.append(dict(start=start,end=end,prompt=[tok.bos_token_id]+prefix+ids[start:end]+suffix,text=tok.decode(ids[start:end])))
        if end==len(ids):break
        start=end-overlap
    assert result and result[0]['start']==0 and result[-1]['end']==len(ids)
    assert all(b['start']<=a['end'] for a,b in zip(result,result[1:]))
    return result

def make_prompt(tok,text,c):
    # Convenience for one excerpt; long-document callers MUST use chunk_document.
    chunks=chunk_document(tok,text,c)
    assert len(chunks)==1, 'Long input: use chunk_document to preserve every source token.'
    return chunks[0]['prompt']

def word_features(text):
    stop=set('the and that this with from were have been which their there court case said was for are not but had has its'.split())
    return set(re.findall(r'[a-z]{3,}',text.lower()))-stop

def aligned_examples(tok,record,c):
    from collections import Counter
    chunks=chunk_document(tok,record['document'],c)
    features=[word_features(ch['text']) for ch in chunks]; df=Counter(w for fs in features for w in fs)
    idf={w:1+math.log((len(chunks)+1)/(n+1)) for w,n in df.items()}
    norms=[math.sqrt(sum(idf[w]**2 for w in fs)) or 1 for fs in features]
    assigned=[[] for ch in chunks]; omitted=0
    sentences=[s.strip() for s in re.split(r'(?<=[.!?])\s+|\n+',record['summary']) if s.strip()]
    for sentence in sentences:
        query=word_features(sentence); qnorm=math.sqrt(sum(idf.get(w,1)**2 for w in query)) or 1
        scores=[sum(idf[w]**2 for w in query & fs)/(norm*qnorm) for fs,norm in zip(features,norms)]
        best=max(range(len(scores)),key=scores.__getitem__)
        if scores[best]>=c['alignment_min_similarity']: assigned[best].append(sentence)
        else: omitted+=1
    examples=[]
    for i,(chunk,gold) in enumerate(zip(chunks,assigned)):
        if not gold:continue
        target,truncated=make_target(tok,' '.join(gold),c['target_tokens']); prompt=chunk['prompt']
        examples.append(dict(id=record['id'],chunk_index=i,x=(prompt+target)[:-1],y=[-100]*(len(prompt)-1)+target,truncated=truncated))
    return examples,dict(chunks=len(chunks),reference_sentences=len(sentences),unaligned_reference_sentences=omitted)

def prediction_tasks(inputs,c):
    tasks=[]
    for index,row in enumerate(inputs):
        n=len(row['chunks'])
        cap=max(c['min_chunk_new_tokens'],min(c['max_chunk_new_tokens'],math.ceil(c['summary_tokens']/n)))
        cap=int(math.ceil(cap/16)*16)
        for ci,ch in enumerate(row['chunks']):tasks.append(dict(index=index,chunk_index=ci,prompt=ch['prompt'],max_new=cap))
    return tasks

def task_batches(tasks,c,rank):
    ordered=sorted(tasks,key=lambda t:(t['max_new'],len(t['prompt']),t['index'],t['chunk_index']))
    mine=ordered[rank::2]; batches=[]
    for task in mine:
        if not batches or len(batches[-1])==c['gen_batch'] or batches[-1][0]['max_new']!=task['max_new']:batches.append([])
        batches[-1].append(task)
    return batches

def merge_chunk_summaries(tok,texts,c):
    candidates=[]; features=[]
    for chunk_i,text in enumerate(texts):
        for sentence in re.split(r'(?<=[.!?])\s+|\n+',text):
            sentence=sentence.strip(); fs=word_features(sentence)
            if not fs:continue
            if any(len(fs & prev)/max(1,len(fs | prev))>=.75 for prev in features):continue
            features.append(fs); candidates.append((chunk_i,len(candidates),sentence,fs))
    assert candidates, 'All chunk generations were empty/unusable'
    # Compress generated sentences only; no source extraction masquerades as model output.
    cues=set('held dismissed allowed appeal liable entitled therefore order judgment constitution section'.split())
    priority=[]
    for i,(ci,order,sentence,fs) in enumerate(candidates):
        score=len(fs & cues)*2+math.log1p(len(fs))
        if i==0:score+=4
        if ci==len(texts)-1:score+=4
        priority.append((score,i))
    chosen=[]; room=c['summary_tokens']
    for _,i in sorted(priority,reverse=True):
        ids=token_ids(tok,candidates[i][2]+' ')
        if len(ids)<=room:chosen.append(i);room-=len(ids)
    if not chosen:
        return tok.decode(token_ids(tok,candidates[0][2])[:c['summary_tokens']]).strip()
    return ' '.join(candidates[i][2] for i in sorted(chosen)).strip()

def assemble_predictions(tok,inputs,shards,c):
    mapping={}
    for p in shards:
        key=(p['index'],p['chunk_index']); assert key not in mapping, 'Duplicate chunk prediction'
        mapping[key]=p['summary']
    expected={(i,ci) for i,r in enumerate(inputs) for ci in range(len(r['chunks']))}
    assert set(mapping)==expected,'Missing or unexpected chunk predictions'
    return [dict(index=i,id=r['id'],split=r['split'],summary=merge_chunk_summaries(tok,[mapping[i,ci] for ci in range(len(r['chunks']))],c)) for i,r in enumerate(inputs)]

def records(path,training):
    obj=json.loads(Path(path).read_text(encoding='utf8'))
    if isinstance(obj,dict):
        obj=next((obj[k] for k in ['data','records','examples'] if isinstance(obj.get(k),list)),obj)
    assert isinstance(obj,list), 'Expected list of records'
    out=[]; seen=set(); dropped=[]
    for r in obj:
        assert isinstance(r,dict) and 'id' in r and isinstance(r.get('document'),str) and r['document'].strip()
        key=str(r['id']); assert key not in seen, 'Duplicate id'; seen.add(key)
        if training and not (isinstance(r.get('summary'),str) and r['summary'].strip()): dropped.append(key); continue
        out.append(dict(r,id=key))
    return out,dropped

def install_fast_mla():
    # Same Phase-2 MLA weights, latent cache, ALiBi and causal/padding masks; fused SDPA for decode too.
    from llama_scratch import MLAttention
    def forward(self,x,cos,sin,alibi_slopes,key_valid,past,use_cache):
        B,T,_=x.shape; q=self.wq(x).view(B,T,self.nh,self.hd).transpose(1,2)
        c=self.kv_norm(self.w_dkv(x))
        if past is not None: c=torch.cat([past,c],dim=1)
        S=c.shape[1]; k=self.w_uk(c).view(B,S,self.nh,self.hd).transpose(1,2); v=self.w_uv(c).view(B,S,self.nh,self.hd).transpose(1,2)
        rel=torch.arange(S-T,S,device=x.device)[:,None]-torch.arange(S,device=x.device)[None,:]
        allowed=(rel>=0)[None,None]
        if key_valid is not None: allowed=allowed & key_valid[:,None,None,:]
        bias=-alibi_slopes.view(1,-1,1,1)*rel.clamp(min=0).float()[None,None]
        mask=bias.masked_fill(~allowed,torch.finfo(q.dtype).min).to(q.dtype)
        out=F.scaled_dot_product_attention(q,k,v,attn_mask=mask)
        return self.wo(out.transpose(1,2).reshape(B,T,self.nh*self.hd)), (c if use_cache else None)
    MLAttention.forward=forward

@torch.inference_mode()
def decode_batch(model,prompts,tok,max_new,force_length=False):
    model.eval(); dev=next(model.parameters()).device; B=len(prompts); L=max(map(len,prompts))
    x=torch.full((B,L),tok.pad_token_id,dtype=torch.long,device=dev); valid=torch.zeros_like(x,dtype=torch.bool)
    for i,p in enumerate(prompts): x[i,L-len(p):]=torch.tensor(p,device=dev); valid[i,L-len(p):]=True
    done=torch.zeros(B,dtype=torch.bool,device=dev); out=[[] for _ in prompts]; seen=torch.zeros(B,model.cfg.vocab_size,dtype=torch.bool,device=dev)
    with torch.autocast(dev.type,dtype=torch.float16,enabled=dev.type=='cuda'):
        logits,cache=model(x,key_valid=valid,use_cache=True)
        for j in range(max_new):
            z=logits[:,-1].float(); z[:,[tok.pad_token_id,tok.bos_token_id,tok.unk_token_id]]=-float('inf')
            if j<32 or force_length: z[:,tok.eos_token_id]=-float('inf')
            z=torch.where(seen,torch.where(z>0,z/1.1,z*1.1),z)
            # Ban repeated generated trigrams without penalizing words copied from the source.
            for b,seq in enumerate(out):
                if len(seq)>=2:
                    banned=[seq[k+2] for k in range(len(seq)-2) if seq[k:k+2]==seq[-2:]]
                    if banned: z[b,banned]=-float('inf')
            nxt=z.argmax(-1); nxt=torch.where(done,torch.full_like(nxt,tok.pad_token_id),nxt)
            values=nxt.tolist()
            for b,v in enumerate(values):
                if not done[b].item() and v!=tok.eos_token_id: out[b].append(v)
            done|=nxt.eq(tok.eos_token_id); seen.scatter_(1,nxt[:,None],True)
            if bool(done.all()) or j+1==max_new: break
            valid=torch.cat([valid,torch.ones(B,1,dtype=torch.bool,device=dev)],dim=1)
            logits,cache=model(nxt[:,None],key_valid=valid,use_cache=True,cache=cache,start_pos=L+j)
    return [tok.decode(s).strip() for s in out]


Download and validate the genuine base/tokenizer; fixed document split; align training targets; prepare every overlapping validation/test window.

In [ ]:
%%writefile phase6_v3_prepare.py
from phase6_v3_common import *
from huggingface_hub import hf_hub_download

def main():
    c=read_config(); out=Path(c['out_dir']); out.mkdir(parents=True,exist_ok=True); token=os.environ.get('HF_TOKEN')
    train_path=hf_hub_download(c['dataset_repo'],c['train_remote'],repo_type='dataset',token=token)
    test_path=hf_hub_download(c['dataset_repo'],c['test_remote'],repo_type='dataset',token=token)
    for f in ['vocab.json','merges.json','tokenizer_config.json']:hf_hub_download(c['tokenizer_repo'],f,local_dir=c['tokenizer_dir'],token=token)
    tok=PureByteLevelBPETokenizer.load(c['tokenizer_dir']); model=load_base(c)
    assert model.cfg.vocab_size==tok.vocab_size
    assert [tok.pad_token_id,tok.unk_token_id,tok.bos_token_id,tok.eos_token_id]==[256,257,258,259]
    train,dropped=records(train_path,True); test,_=records(test_path,False); assert len(train)>2 and test
    random.Random(c['seed']).shuffle(train); nv=max(2,round(len(train)*.05)); val=train[:nv]; train=train[nv:]
    audits=[]
    def examples(rows):
        result=[]
        for i,r in enumerate(rows):
            ex,audit=aligned_examples(tok,r,c); result.extend(ex); audits.append(audit)
            if (i+1)%500==0:print('Aligned',i+1,'documents;',len(result),'chunk examples',flush=True)
        return result
    tx=examples(train); vx=examples(val[:c['val_loss_docs']])
    assert len(tx)>=32 and vx,'Too few aligned examples. Inspect references and alignment threshold.'
    random.Random(c['seed']).shuffle(vx);vx=vx[:c['val_loss_chunks']]
    dev=torch.device('cuda:0'); model=model.to(dev).eval(); ids=token_ids(tok,train[0]['document'])[:256]
    with torch.no_grad(),torch.autocast('cuda',dtype=torch.float16):sanity=float(model(torch.tensor([ids[:-1]],device=dev),targets=torch.tensor([ids[1:]],device=dev)))
    assert math.isfinite(sanity) and sanity<8,f'LM sanity failed ({sanity}); check genuine v3 weights/tokenizer.'
    torch.save(dict(train=tx,val=vx),out/'prepared.pt')
    inference=[]
    for split,rows in [('validation',val[:c['val_gen_docs']]),('test',test)]:
        for r in rows:
            chunks=chunk_document(tok,r['document'],c)
            inference.append(dict(id=r['id'],split=split,chunks=[dict(start=ch['start'],end=ch['end'],prompt=ch['prompt']) for ch in chunks],reference=r.get('summary')))
    torch.save(inference,out/'inference_inputs.pt')
    report=dict(base_cfg=asdict(model.cfg),lm_sanity_loss=sanity,train_docs=len(train),train_chunk_examples=len(tx),validation_docs=len(val),validation_chunk_examples=len(vx),test_docs=len(test),test_chunks=sum(len(r['chunks']) for r in inference if r['split']=='test'),dropped_empty_ids=dropped,target_truncation_fraction=sum(e['truncated'] for e in tx)/len(tx),alignment=dict(method='reference sentence to best source chunk by source-local IDF cosine; training/validation CE only',reference_sentences=sum(a['reference_sentences'] for a in audits),unaligned_reference_sentences=sum(a['unaligned_reference_sentences'] for a in audits)),coverage='All source tokens covered by overlapping windows; all inference chunks scheduled',chunk_overlap=c['chunk_overlap'])
    atomic_json(report,out/'data_audit.json');print(json.dumps(report,indent=2),flush=True)

if __name__=='__main__':main()


Two-process DDP, decode-time benchmark, budgeted LoRA teacher training followed by online soft-target student distillation.

In [ ]:
%%writefile phase6_v3_train.py
import contextlib, gc
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from phase6_v3_common import *

def setup(c):
    rank=int(os.environ['LOCAL_RANK']); torch.cuda.set_device(rank); dist.init_process_group('nccl')
    assert dist.get_world_size()==2, 'Select Kaggle GPU T4 x2, not one GPU.'
    seed_all(c['seed']); torch.set_num_threads(2)
    return rank,torch.device('cuda',rank)

def max_time():
    t=torch.tensor(time.time(),device=torch.cuda.current_device(),dtype=torch.float64); dist.all_reduce(t,op=dist.ReduceOp.MAX); return t.item()

def evaluate(model,examples,c,device,rank):
    model.eval(); sums=torch.zeros(2,device=device,dtype=torch.float64)
    with torch.no_grad():
        for i in range(rank,len(examples),2):
            x,y=collate([examples[i]],c['pad'],device)
            with torch.autocast('cuda',dtype=torch.float16): ce,_=model(x,y)
            count=y.ne(-100).sum(); sums[0]+=ce.double()*count; sums[1]+=count
    dist.all_reduce(sums); value=(sums[0]/sums[1]).item(); assert math.isfinite(value)
    return value

def train_stage(model,teacher,data,c,rank,device,stage,deadline,max_steps):
    out=Path(c['out_dir']); params=[p for p in model.parameters() if p.requires_grad]
    obj=Objective(model).to(device); ddp=DDP(obj,device_ids=[rank],broadcast_buffers=False)
    opt=torch.optim.AdamW(params,lr=c['lora_lr'] if stage=='lora' else c['student_lr'],weight_decay=.01)
    for group in opt.param_groups: group['lr'] /= 20  # warmup starts on the first update
    scaler=torch.amp.GradScaler('cuda',init_scale=1024)
    baseline=evaluate(obj,data['val'],c,device,rank); best=float('inf'); rows=[]
    if rank==0: print(f'{stage}: baseline val={baseline:.4f}, trainable={sum(p.numel() for p in params):,}',flush=True)
    rng=random.Random(c['seed']); successful=0; attempts=0; skipped=0; epoch=0; pointer=0; order=[]; start=time.time()
    # Monitor the zero-initialized LoRA B, or a representative student projection.
    probe=next(p for n,p in model.named_parameters() if p.requires_grad and (n.endswith('.B') if stage=='lora' else 'attn.wq.weight' in n))
    original=probe.detach().clone(); n=len(data['train']); bs=c['micro_batch']; accum=c['grad_accum']
    while successful<max_steps:
        if max_time()>deadline-45: break
        if pointer+2*bs*accum>len(order):
            order=list(range(n)); rng.shuffle(order); pointer=0; epoch+=1
            # Extend only the final incomplete global batch, deterministically on both ranks.
            multiple=2*bs*accum; order+=(order*((multiple+n-1)//n))[:(-len(order))%multiple]
        ddp.train(); opt.zero_grad(set_to_none=True); ce_sum=kd_sum=0.
        for micro in range(accum):
            at=pointer+micro*2*bs+rank*bs; ex=[data['train'][i] for i in order[at:at+bs]]
            x,y=collate(ex,c['pad'],device); nt=int(y.ne(-100).sum()); positions=None; tz=None
            if teacher is not None:
                positions=torch.linspace(0,nt-1,min(nt,c['kd_positions']),device=device).long()
                with torch.no_grad(),torch.autocast('cuda',dtype=torch.float16):
                    th=hidden(teacher,x)[y.ne(-100)][positions]; tz=teacher.lm_head(th).float()
            sync=ddp.no_sync() if micro+1<accum else contextlib.nullcontext()
            with sync:
                with torch.autocast('cuda',dtype=torch.float16):
                    ce,sz=ddp(x,y,positions); loss=ce; kd=ce.detach()*0
                    if teacher is not None:
                        T=c['temperature']; kd=F.kl_div(F.log_softmax(sz/T,dim=-1),F.softmax(tz/T,dim=-1),reduction='batchmean')*(T*T)
                        loss=c['ce_weight']*ce+(1-c['ce_weight'])*kd
                finite=torch.tensor(int(torch.isfinite(loss)),device=device); dist.all_reduce(finite,op=dist.ReduceOp.MIN)
                assert finite.item(), f'{stage}: nonfinite loss; stopped before corrupting weights'
                scaler.scale(loss/accum).backward()
            ce_sum+=float(ce.detach())/accum; kd_sum+=float(kd.detach())/accum
        pointer+=2*bs*accum; attempts+=1; scaler.unscale_(opt)
        norm=torch.nn.utils.clip_grad_norm_(params,1.)
        # DDP synchronizes gradients; both ranks must agree on whether AMP skipped a step.
        old=scaler.get_scale(); scaler.step(opt); scaler.update(); did=int(scaler.get_scale()>=old)
        flag=torch.tensor([did,did],device=device); dist.all_reduce(flag[:1],op=dist.ReduceOp.MIN); dist.all_reduce(flag[1:],op=dist.ReduceOp.MAX)
        assert flag[0]==flag[1], 'AMP update diverged between ranks'
        if did:
            successful+=1
            warm=min(1.,successful/20)
            progress=min(1.,max(successful/max_steps,(time.time()-start)/max(1.,deadline-start-45)))
            decay=max(.1,.5*(1+math.cos(math.pi*progress)))
            for g in opt.param_groups:g['lr']=(c['lora_lr'] if stage=='lora' else c['student_lr'])*warm*decay
            if successful==1: assert (probe.detach()-original).abs().max()>0, 'Optimizer did not update the monitored weight'
        else:
            skipped+=1
            assert skipped<20, 'Too many AMP overflows: inspect checkpoint and numeric health'
        if rank==0 and (attempts%10==0 or successful==1):
            print(f'{stage} updates={successful} attempted={attempts} skipped={skipped} CE={ce_sum:.4f} KD={kd_sum:.4f} grad={float(norm):.3f} minutes={(time.time()-start)/60:.1f}',flush=True)
        due=successful and did and (successful%c['eval_every']==0 or successful==max_steps)
        if due:
            val=evaluate(obj,data['val'],c,device,rank); delta=float((probe.detach()-original).abs().max())
            row=dict(stage=stage,successful_updates=successful,attempted_updates=attempts,skipped=skipped,train_ce=ce_sum,train_kd=kd_sum,val_ce=val,weight_max_change=delta,seconds=time.time()-start)
            if rank==0:
                rows.append(row); print(row,flush=True)
                if val<best:
                    # Save adapters in training state; the export later merges them exactly once.
                    save_model(model,out/f'{stage}_best_training.pt',stage=stage,metrics=row,baseline_val=baseline)
                atomic_json(rows,out/f'{stage}_history.json')
            best=min(best,val); dist.barrier()
    assert successful>=c['min_updates'], f'{stage}: only {successful} updates; budget insufficient for a valid run'
    val=evaluate(obj,data['val'],c,device,rank)
    row=dict(stage=stage,successful_updates=successful,attempted_updates=attempts,skipped=skipped,val_ce=val,weight_max_change=float((probe.detach()-original).abs().max()),baseline_val=baseline,seconds=time.time()-start)
    if rank==0:
        rows.append(row)
        if val<best: save_model(model,out/f'{stage}_best_training.pt',stage=stage,metrics=row,baseline_val=baseline)
        atomic_json(rows,out/f'{stage}_history.json')
    dist.barrier(); del ddp,opt,scaler; gc.collect(); torch.cuda.empty_cache()
    ck=torch.load(out/f'{stage}_best_training.pt',map_location='cpu',weights_only=False); model.load_state_dict(ck['model'],strict=True)
    best_val=min(best,val)
    if rank==0:
        print(f'{stage} validation CE: {baseline:.4f} -> {best_val:.4f}; improvement={baseline-best_val:.4f}',flush=True)
    return model,dict(final=row,best_val=best_val,baseline_val=baseline)

def main():
    c=read_config(); rank,device=setup(c); out=Path(c['out_dir']); install_fast_mla()
    data=torch.load(out/'prepared.pt',weights_only=False); tok=PureByteLevelBPETokenizer.load(c['tokenizer_dir']); c['pad']=tok.pad_token_id
    teacher=load_base(c).to(device); prompts=[e['x'][:next(i for i,v in enumerate(e['y']) if v!=-100)+1] for e in sorted(data['val'],key=lambda e:len(e['x']),reverse=True)[:c['gen_batch']]]
    torch.cuda.synchronize(); t=time.time(); decode_batch(teacher,prompts,tok,64,force_length=True); torch.cuda.synchronize()
    bench=torch.tensor(time.time()-t,device=device); dist.all_reduce(bench,op=dist.ReduceOp.MAX)
    inputs=torch.load(out/'inference_inputs.pt',weights_only=False)
    # Every source chunk participates in the base comparison as well as trained inference.
    val_tasks=[t for t in prediction_tasks(inputs,c) if inputs[t['index']]['split']=='validation']; baseline=[]
    for batch in task_batches(val_tasks,c,rank):
        texts=decode_batch(teacher,[t['prompt'] for t in batch],tok,batch[0]['max_new'])
        baseline.extend(dict(index=t['index'],chunk_index=t['chunk_index'],summary=text) for t,text in zip(batch,texts))
    atomic_json(baseline,out/f'baseline_rank{rank}.json');dist.barrier()
    tasks=prediction_tasks(inputs,c)
    # Benchmark already includes prefill + 64 decoding steps. Every task batch is accounted for.
    workload=max(sum(batch[0]['max_new']/64 for batch in task_batches(tasks,c,r)) for r in [0,1])
    reserve=max(12*60,float(bench)*workload*2*1.5+180)
    remaining=c['started_at']+c['total_minutes']*60-time.time()
    assert remaining-reserve>12*60, f'Benchmark leaves insufficient training time: generation reserve {reserve/60:.1f}m, remaining {remaining/60:.1f}m. Shorten context/output explicitly or use faster hardware.'
    training_end=c['started_at']+c['total_minutes']*60-reserve
    teacher_end=time.time()+.55*(training_end-time.time())
    if rank==0: atomic_json(dict(decode_64_batch_seconds=float(bench),inference_reserve_minutes=reserve/60,total_inference_chunks=len(tasks),teacher_deadline=teacher_end,student_deadline=training_end),out/'budget_plan.json')
    names=inject_lora(teacher,c); teacher=teacher.to(device)
    teacher,tmetrics=train_stage(teacher,None,data,c,rank,device,'lora',teacher_end,c['teacher_steps'])
    teacher=merge_lora(teacher).eval().requires_grad_(False)
    if rank==0: save_model(teacher,out/'lora_teacher.pt',stage='merged_lora',metrics=tmetrics,lora_targets=names)
    dist.barrier(); student,selected=student_from_teacher(teacher,c); student=student.to(device)
    student,smetrics=train_stage(student,teacher,data,c,rank,device,'student',training_end,c['student_steps'])
    if rank==0:
        save_model(student,out/'distilled_student.pt',stage='distilled_student',metrics=smetrics,teacher_layers=selected)
        atomic_json(dict(teacher=tmetrics,student=smetrics,student_teacher_layer_map=selected,teacher_parameters=sum(p.numel() for p in teacher.parameters()),student_parameters=sum(p.numel() for p in student.parameters()),kd=dict(temperature=c['temperature'],reference_ce_weight=c['ce_weight'],soft_target_positions_per_microbatch=c['kd_positions'])),out/'training_metrics.json')
    dist.barrier(); dist.destroy_process_group()

if __name__=='__main__':main()


Both GPUs summarize every overlapping source chunk for both models. Full chunk coverage is checked before document-level prediction export.

In [ ]:
%%writefile phase6_v3_predict.py
import torch.distributed as dist
from phase6_v3_common import *

def main():
    c=read_config();rank=int(os.environ['LOCAL_RANK']);torch.cuda.set_device(rank);dist.init_process_group('nccl');assert dist.get_world_size()==2
    install_fast_mla();tok=PureByteLevelBPETokenizer.load(c['tokenizer_dir']);out=Path(c['out_dir']);rows=torch.load(out/'inference_inputs.pt',weights_only=False)
    device=torch.device('cuda',rank);batches=task_batches(prediction_tasks(rows,c),c,rank)
    for stage,file in [('lora','lora_teacher.pt'),('student','distilled_student.pt')]:
        model=load_saved(out/file).to(device).eval();predictions=[]
        for at,batch in enumerate(batches):
            assert time.time()<c['started_at']+c['total_minutes']*60-30,'Deadline reached; partial shards retained in /tmp, no completed official submission claimed.'
            texts=decode_batch(model,[t['prompt'] for t in batch],tok,batch[0]['max_new'])
            for t,text in zip(batch,texts):
                assert text.strip(),f'Empty generation for document {rows[t["index"]]["id"]}, chunk {t["chunk_index"]}'
                predictions.append(dict(index=t['index'],chunk_index=t['chunk_index'],summary=text))
            atomic_json(predictions,out/f'{stage}_rank{rank}_predictions.json')
            if at%5==0 or at+1==len(batches):print(f'{stage} GPU{rank}: {at+1}/{len(batches)} chunk batches',flush=True)
        del model;torch.cuda.empty_cache();dist.barrier()
    dist.destroy_process_group()

if __name__=='__main__':main()


Verify complete coverage, compare held-out generation against the untuned base, create one prediction-only folder and ZIP per model, then remove temporary weights.

In [ ]:
%%writefile phase6_v3_finalize.py
from phase6_v3_common import *
import zipfile, shutil, tempfile
from rouge_score import rouge_scorer

def finalize(c):
    runtime=Path(c['out_dir']); out=Path(c['submission_dir']); out.mkdir(parents=True,exist_ok=True)
    inputs=torch.load(runtime/'inference_inputs.pt',weights_only=False); metrics={}
    tok=PureByteLevelBPETokenizer.load(c['tokenizer_dir'])
    scorer=rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'],use_stemmer=True)
    def score(pairs):
        assert pairs, 'No held-out references'
        scores=[scorer.score(a,b) for a,b in pairs]
        return {key:sum(s[key].fmeasure for s in scores)/len(scores) for key in ['rouge1','rouge2','rougeL']}
    base=[]
    for rank in [0,1]:base+=json.loads((runtime/f'baseline_rank{rank}.json').read_text(encoding='utf8'))
    val_indices=[i for i,r in enumerate(inputs) if r['split']=='validation']
    remap={old:new for new,old in enumerate(val_indices)}
    base=[dict(p,index=remap[p['index']]) for p in base]
    baseline_docs=assemble_predictions(tok,[inputs[i] for i in val_indices],base,c)
    metrics['untuned_base']=score([(inputs[i]['reference'],p['summary']) for i,p in zip(val_indices,baseline_docs)])
    for stage in ['lora','student']:
        pred=[]
        for rank in [0,1]:pred+=json.loads((runtime/f'{stage}_rank{rank}_predictions.json').read_text(encoding='utf8'))
        pred=assemble_predictions(tok,inputs,pred,c)
        test=[dict(id=p['id'],summary=p['summary']) for p in pred if p['split']=='test']
        assert len(test)==len({r['id'] for r in test})==sum(r['split']=='test' for r in inputs)
        assert all(isinstance(p['id'],str) and isinstance(p['summary'],str) and p['summary'].strip() for p in test)
        model_folder=out/f'submission_{stage}'; model_folder.mkdir(exist_ok=True)
        prediction=model_folder/'prediction.json'; atomic_json(test,prediction)
        archive=out/f'submission_{stage}.zip'
        with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
            z.write(prediction,arcname='prediction.json')
        with zipfile.ZipFile(archive) as z:
            assert z.namelist()==['prediction.json']
            assert json.loads(z.read('prediction.json'))==test
        metrics[stage]=score([(r['reference'],p['summary']) for p,r in zip(pred,inputs) if r['split']=='validation'])
        metrics[stage]['test_documents']=len(test)
        metrics[stage]['rougeL_gain_over_base']=metrics[stage]['rougeL']-metrics['untuned_base']['rougeL']
    selected=max(['lora','student'],key=lambda s:metrics[s]['rougeL'])
    print('SAME-REFERENCE HELD-OUT GENERATION COMPARISON (not test metrics):',flush=True)
    print(json.dumps(metrics,indent=2),flush=True)
    print('Higher held-out ROUGE-L:',selected,flush=True)
    for stage in ['lora','student']:
        if metrics[stage]['rougeL_gain_over_base']<=0:
            print(f'QUALITY CHECK: {stage} did not improve held-out ROUGE-L over the untuned base. Do not claim improved quality.',flush=True)
    training=json.loads((runtime/'training_metrics.json').read_text(encoding='utf8'))
    print('TRAINING EVIDENCE:',json.dumps(training,indent=2),flush=True)
    print('Elapsed minutes:',round((time.time()-c['started_at'])/60,2),flush=True)
    if c.get('cleanup_runtime',False):
        # Delete only this task's fixed, resolved temporary directory after both ZIPs verify.
        resolved=runtime.resolve(); expected=Path(tempfile.gettempdir()).resolve()/'phase6_v3_v33_runtime'
        assert resolved==expected and resolved.parent==Path(tempfile.gettempdir()).resolve(), f'Refusing unsafe cleanup: {resolved}'
        assert not Path(c['checkpoint']).resolve().is_relative_to(resolved), 'Base input must remain outside temporary outputs'
        shutil.rmtree(resolved)
        assert not resolved.exists()
        print('Temporary checkpoints, caches and shards removed. Final outputs contain predictions only.',flush=True)
    print('Saved predictions:',out,flush=True)
    return metrics

def main():finalize(read_config())
if __name__=='__main__':main()


In [ ]:
# Prepare data and validate the genuine Phase 2 v3 base
run_checked([sys.executable, '-u', 'phase6_v3_prepare.py'], 'prepare.log')


In [ ]:
# Train LoRA teacher then distill student on BOTH GPUs
run_checked([sys.executable, '-u', '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', 'phase6_v3_train.py'], 'training.log')


In [ ]:
# Generate complete teacher and student predictions on BOTH GPUs
run_checked([sys.executable, '-u', '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', 'phase6_v3_predict.py'], 'inference.log')


In [ ]:
# Package prediction.json for each model and remove temporary weights
run_checked([sys.executable, '-u', 'phase6_v3_finalize.py'], 'finalize.log')
from IPython.display import display, FileLink
import zipfile
out = Path(CONFIG['submission_dir'])
for model_name in ['lora', 'student']:
    prediction = out/f'submission_{model_name}'/'prediction.json'
    archive = out/f'submission_{model_name}.zip'
    with zipfile.ZipFile(archive) as z:
        assert z.namelist() == ['prediction.json']
    display(FileLink(str(prediction)))
    display(FileLink(str(archive)))
assert not list(out.rglob('*.pt')), 'No model weights should be exported.'
print(f'Completed in {(time.time()-RUN_STARTED_AT)/60:.2f} minutes.')
assert time.time()-RUN_STARTED_AT < 90*60, 'Complete predictions exist, but this execution exceeded the requested 90 minutes.'


In [ ]:
# Copy everything into /kaggle/working, then upload to Hugging Face under phase6_v3
import shutil, os
from pathlib import Path
from huggingface_hub import HfApi

REPO_ID = "DumbLad/legal-phase6-v3"   # change to your own username/repo
TOKEN = os.environ.get("HF_WRITE_TOKEN")
if not TOKEN:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("HF_WRITE")   # Kaggle secret with a WRITE token

# Weights, histories, metrics logs, data audit, budget plan, tokenizer
art = Path("/kaggle/working/phase6_v3_artifacts")
if art.exists(): shutil.rmtree(art)
shutil.copytree(CONFIG["out_dir"], art)

api = HfApi(token=TOKEN)
api.create_repo(REPO_ID, repo_type="model", private=True, exist_ok=True)
api.upload_folder(
    folder_path="/kaggle/working",
    path_in_repo="phase6_v3",
    repo_id=REPO_ID,
    repo_type="model",
    ignore_patterns=["__pycache__/*", ".ipynb_checkpoints/*", "*.tmp"],
    commit_message="Phase 6 v3 v33: predictions, weights, logs",
)
print("Uploaded to https://huggingface.co/" + REPO_ID + "/tree/main/phase6_v3")

## Interpretation and long-text limits
All source tokens occur in at least one overlapping input window. The model generates a summary for **every** window. Duplicate generated sentences are removed, then salient generated sentences are selected within a 768-token document budget, with extra weight for the first generated sentence and final-window outcome. Selected sentences retain their original order. This handles inputs longer than one model context without changing the base architecture. Cross-chunk reasoning, heuristic target alignment and missing details in a short final summary remain limitations.

Per-chunk output caps adapt to document chunk count (48–192 tokens, rounded to batches of 16). The output-length budget is not a cap on source coverage. All chunks are scheduled and completeness is checked before packaging. Both GPUs shard chunk generation. The runtime benchmark accounts for all chunk batches, rather than estimating from document count alone.

The printed held-out comparison includes the untuned base with the same chunking/reducer. A zero or negative `rougeL_gain_over_base` prints a quality warning. Neither lower CE nor successful tiny-model tests guarantee better actual summaries. Distillation may reduce quality; both submissions are preserved. Metrics are computed on 16 training-derived validation documents, not test references.

No external RAG system is used. The original RAG approach retrieves external knowledge for knowledge-intensive tasks ([Lewis et al.](https://arxiv.org/abs/2005.11401)); splitting long inputs for summarization has direct precedent ([SUMM^N](https://arxiv.org/abs/2110.10150)). This notebook's budgeted generated-sentence reducer is its own simpler implementation, not a reproduction of SUMM^N.

Download `submission_lora.zip` and `submission_student.zip`. Each contains one root member named `prediction.json`. The matching model folder contains the same file. On success, temporary model weights and caches are deleted.
